# Severity pipeline training

Export the same TF-IDF and logistic-regression baseline used by the application. The exported bundle is for inspected local training experiments; the API currently fits the bundled examples in memory and does not load this pickle file.

In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "ResQ_Kerala_Complete_Project_Discussion.md").is_file())
sys.path.insert(0, str(PROJECT_ROOT))
from backend.ai.severity_model import load_examples, new_pipeline, DISCLAIMER
rows = load_examples()
print(DISCLAIMER)


Synthetic training examples; uncalibrated demonstration score; admin verification required.


In [2]:
import json
import platform
import joblib
import sklearn
pipeline = new_pipeline().fit([row["message"] for row in rows], [row["severity"] for row in rows])
artifact_dir = PROJECT_ROOT / "models" / "severity_classifier"
artifact_dir.mkdir(parents=True, exist_ok=True)
joblib.dump(pipeline, artifact_dir / "pipeline.joblib")
metadata = {"training_source": "synthetic_demo", "training_count": len(rows),
            "notice": DISCLAIMER, "operational_validation": False,
            "python_version": platform.python_version(), "sklearn_version": sklearn.__version__,
            "artifact": "models/severity_classifier/pipeline.joblib", "serving_mode": "in_memory_training"}
(artifact_dir / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
asset_dir = PROJECT_ROOT / "backend" / "ml_models"
for folder in ("severity_model", "tokenizer", "preprocessing"):
    (asset_dir / folder).mkdir(parents=True, exist_ok=True)
(asset_dir / "severity_model" / "manifest.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
vectorizer = pipeline.named_steps["tfidf"]
vocabulary = {token: int(index) for token, index in vectorizer.vocabulary_.items()}
(asset_dir / "tokenizer" / "vocabulary.json").write_text(json.dumps(vocabulary, indent=2, ensure_ascii=False), encoding="utf-8")
(asset_dir / "preprocessing" / "config.json").write_text(json.dumps(vectorizer.get_params(), indent=2, default=str), encoding="utf-8")
print({"artifact": metadata["artifact"], "training_count": len(rows), "vocabulary_size": len(vocabulary)})


{'artifact': 'models/severity_classifier/pipeline.joblib', 'training_count': 60, 'vocabulary_size': 693}


Load joblib bundles only from a trusted source and a compatible pinned environment. This notebook records the Python and scikit-learn versions. Use the evaluation notebook for held-out metrics; training-set performance is not validation.